<a href="https://colab.research.google.com/github/preethi-06r/SepsisGuard-AI/blob/main/Copy_of_SepsisGuard_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install necessary libraries
!pip install shap xgboost gradio imbalanced-learn -q

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import gradio as gr
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

# 2. Logic-Based Clinical Data Generator
# This creates data where high HR and low O2 actually lead to Sepsis (Better for ROC-AUC)
def generate_clinical_data(samples=3000):
    np.random.seed(42)
    hr = np.random.normal(80, 15, samples)
    o2 = np.random.normal(97, 3, samples)
    temp = np.random.normal(37, 1, samples)
    sbp = np.random.normal(120, 20, samples)
    map_val = np.random.normal(80, 12, samples)
    resp = np.random.normal(18, 5, samples)
    age = np.random.randint(18, 90, samples)

    # Logic: If HR > 100 and O2 < 90, higher chance of Sepsis
    sepsis_prob = (hr > 100).astype(int) + (o2 < 92).astype(int) + (temp > 38).astype(int)
    sepsis_label = (sepsis_prob >= 2).astype(int)

    data = {
        'HR': hr, 'O2Sat': o2, 'Temp': temp, 'SBP': sbp,
        'MAP': map_val, 'Resp': resp, 'Age': age, 'SepsisLabel': sepsis_label
    }
    return pd.DataFrame(data)

df = generate_clinical_data()

# 3. Preprocessing & Balancing
features = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp', 'Age']
X = df[features]
y = df['SepsisLabel']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Handling Data Imbalance (SMOTE)
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

# 4. Optimized Model Training
model = xgb.XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=6,
    eval_metric='logloss', # Fixes UserWarning
    random_state=42
)
model.fit(X_res, y_res)

# 5. Model Evaluation
y_pred_proba = model.predict_proba(X_test)[:, 1]
score = roc_auc_score(y_test, y_pred_proba)
print(f"✅ Model Trained Successfully!")
print(f"🎯 Improved ROC-AUC Score: {score:.2f}")

# 6. Explainable AI (SHAP) - Visualizing 1 prediction
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# 7. Deployment (Gradio UI)
def predict_sepsis(HR, O2Sat, Temp, SBP, MAP, Resp, Age):
    input_df = pd.DataFrame([[HR, O2Sat, Temp, SBP, MAP, Resp, Age]], columns=features)
    prob = model.predict_proba(input_df)[0][1]

    # Logic for status
    if prob > 0.7: status = "🚨 CRITICAL: High Risk of Sepsis"
    elif prob > 0.4: status = "⚠️ WARNING: Moderate Risk"
    else: status = "✅ STABLE: Low Risk"

    return f"Sepsis Probability: {prob:.2f}\nStatus: {status}"

interface = gr.Interface(
    fn=predict_sepsis,
    inputs=[gr.Number(label="Heart Rate (bpm)"), gr.Number(label="O2 Saturation (%)"),
            gr.Number(label="Body Temp (°C)"), gr.Number(label="Systolic BP"),
            gr.Number(label="MAP"), gr.Number(label="Resp Rate"), gr.Number(label="Age")],
    outputs="text",
    title="SepsisGuard-AI: Intelligent Clinical Monitoring",
    description="Early detection prototype for the AI4Dev '26 National Hackathon. TRL Level 3."
)

interface.launch(share=True)

✅ Model Trained Successfully!
🎯 Improved ROC-AUC Score: 1.00
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://70eb9fcee5f1072091.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
